# Oblique whistler waves in Hall MHD

A uniform plasma (density $\rho_0$, pressure $p_0$, adiabatic exponent $\gamma$) is threaded by
the uniform field $\mathbf B_0=B_0\hat{\mathbf z}$. In Hall magnetohydrodynamics the electric
field carries, besides the ideal part $-\mathbf v\times\mathbf B$, the Hall term

$$\mathbf E_{\rm Hall}=\eta_H\,\frac{\mathbf J\times\mathbf B}{\rho},\qquad
  \mathbf J=\nabla\times\mathbf B\qquad(\mu_0=1),$$

which decouples the ions from the field on scales below the ion inertial length
$d_i=\eta_H/\sqrt{\rho_0}$. The shear-Alfvén wave then splits into a right-hand **whistler**
branch whose frequency grows like $k^2$, and a left-hand **ion-cyclotron / kinetic-Alfvén-type**
branch. Both depend on the angle $\theta$ between the wave vector $\mathbf k$ and $\mathbf B_0$.

**Exact reference.** Linearising the compressible, adiabatic Hall-MHD equations about the uniform
state gives the dispersion relation for oblique propagation (E. Hameiri, A. Ishizawa and
A. Ishida, *Waves in the Hall-magnetohydrodynamics model*, Phys. Plasmas 12, 072109, 2005)

$$\left(\omega^2-k_\parallel^2v_A^2\right)\left(\omega^4-k^2(v_A^2+c_s^2)\,\omega^2+k^2k_\parallel^2v_A^2c_s^2\right)
  =\omega^2\,k_\parallel^2k^2\,v_A^2\,d_i^2\left(\omega^2-k^2c_s^2\right),$$

with $k_\parallel=k\cos\theta$, the Alfvén speed $v_A=B_0/\sqrt{\rho_0}$ and the sound speed
$c_s=\sqrt{\gamma p_0/\rho_0}$. Its three positive roots are the slow, the ion-cyclotron /
kinetic-Alfvén-type and the fast / whistler branch. For $d_i=0$ it factorises into the ideal
Alfvén and magnetosonic waves; for $\theta=0$ it reduces to the parallel whistler
$\omega=kv_A[\sqrt{1+(kd_i/2)^2}+kd_i/2]$ and the ion-cyclotron wave with the minus sign; for
large $k$ the whistler approaches $\omega\simeq k\,k_\parallel d_i v_A$. The eigenvectors of the
same linear system give the polarisation of each branch.

**Numerical method.** OneSimulation's three-dimensional resistive/Hall-MHD solver integrates the
eight conservative fields (density, three momenta, total energy, three magnetic components) in
conservative form. Two spatial discretisations of the same equations are compared on a periodic
collocation grid:

* Fourier pseudo-spectral derivatives (exact for every resolved Fourier mode),
* second-order central finite differences, whose derivative of $e^{ikx}$ is $i\tilde k\,e^{ikx}$
  with the modified wavenumber $\tilde k=\sin(kh)/h$.

Time integration is the explicit classical fourth-order Runge-Kutta method (RK4).

**What is checked.** Frequencies of all launched modes against the exact dispersion relation;
polarisation; total-energy conservation and amplitude preservation of every mode; a resolution
study for both spatial schemes (with the modified-wavenumber prediction for the difference
scheme); and a time-step study with the observed order of RK4.

In [ ]:
from importlib.metadata import version
import time
import numpy as np

from onesim.physics.magnetohydrodynamics import ResistiveMHD3DSolver
from onesim.results import fourier_coefficient, fourier_interpolate, oscillation_fit
from onesim.visualization import theme, plots

theme.set()
print("OneSimulation:", version("onesimulation"))

## Parameters

Dimensionless units with $\mu_0=1$: $\rho_0=1$, $B_0=1$ (so $v_A=1$), $\eta_H=0.5$ (so
$d_i=0.5$), $\gamma=5/3$ and $c_s=0.5$ (plasma beta $2p_0/B_0^2=0.3$). The periodic box has the
side $2\pi$, so the admissible wave vectors are integer triples $(k_x,k_y,k_z)$.

Eight modes are launched on the whistler branch and six on the ion-cyclotron / kinetic-Alfvén
branch, each with a magnetic amplitude of $10^{-7}B_0$ (small enough that the quadratic coupling
between the harmonics $\mathbf k$, $2\mathbf k$, $3\mathbf k$ stays below the time-integration error): a family with $|\mathbf k|=3$ at
$\theta=0^\circ$, $48.2^\circ$, $70.5^\circ$ and $90^\circ$, and families along $\theta=0^\circ$
and $\theta=45^\circ$ with $|\mathbf k|$ between 1 and 4.24 ($kd_i$ up to 2.1).

In [ ]:
rho_0, B_0, eta_H, gamma, c_s = 1.0, 1.0, 0.5, 5.0 / 3.0, 0.5
p_0 = rho_0 * c_s**2 / gamma
B0_vec = np.array([0.0, 0.0, B_0])
v_A, d_i = B_0 / np.sqrt(rho_0), eta_H / np.sqrt(rho_0)
amplitude = 1.0e-7
final_time = 4.0 * np.pi
sample_interval = 0.08

WHISTLER, ION_CYCLOTRON = -1, -2
modes = [((0, 0, 1), WHISTLER), ((0, 0, 3), WHISTLER), ((1, 0, 1), WHISTLER), ((2, 0, 2), WHISTLER),
         ((3, 0, 3), WHISTLER), ((1, 2, 2), WHISTLER), ((2, 2, 1), WHISTLER), ((0, 3, 0), WHISTLER),
         ((0, 0, 2), ION_CYCLOTRON), ((0, 1, 1), ION_CYCLOTRON), ((0, 2, 2), ION_CYCLOTRON),
         ((0, 3, 3), ION_CYCLOTRON), ((2, 1, 2), ION_CYCLOTRON), ((2, -2, 1), ION_CYCLOTRON)]
branch_name = {WHISTLER: "whistler", ION_CYCLOTRON: "ion-cyclotron/KAW"}
print(f"v_A = {v_A:.3f}, d_i = {d_i:.3f}, c_s = {c_s:.3f}, beta = {2*p_0/B_0**2:.3f}")

## Geometry and mesh

The domain is the cube $[0,2\pi)^3$, periodic in $x$, $y$ and $z$ on all six faces (the solver's
only boundary condition). The grids have $N^3$ points with $N=8$, 16 and 32 for the spectral
scheme and $N=16$, 32 and 64 for the difference scheme; the main run uses the spectral scheme
on $32^3$ points. All fourteen modes are superposed, so the field is a genuinely
three-dimensional interference pattern.

## Model

**Linear theory.** For a plane wave $\propto e^{i\mathbf k\cdot\mathbf x}$ the linearised equations for
$X=(\rho',\mathbf v,p',\mathbf b)$ read $\dot X=L(\mathbf k)X$ with

$$\dot\rho'=-\rho_0\,i\mathbf k\cdot\mathbf v,\quad
  \rho_0\dot{\mathbf v}=-i\mathbf k\,p'-i\mathbf k(\mathbf B_0\cdot\mathbf b)+i(\mathbf k\cdot\mathbf B_0)\mathbf b,\quad
  \dot p'=-\gamma p_0\,i\mathbf k\cdot\mathbf v,$$
$$\dot{\mathbf b}=i\mathbf k\times(\mathbf v\times\mathbf B_0)-\frac{\eta_H}{\rho_0}\,i\mathbf k\times\big[(i\mathbf k\times\mathbf b)\times\mathbf B_0\big].$$

The eigenvalues of $L$ are $-i\omega$; the positive frequencies agree with the three roots of the
closed-form relation above (both are printed below), and the eigenvectors supply the initial
state of each launched mode. For the difference scheme every derivative of a Fourier mode is
multiplied by $i\tilde{\mathbf k}$ instead of $i\mathbf k$, so its exact discrete frequency is
$\omega(\tilde{\mathbf k})$ with $\tilde k_j=\sin(k_jh)/h$; the difference runs are launched with
the eigenvectors of $L(\tilde{\mathbf k})$ so that each excites a single discrete branch.

In [ ]:
def linear_operator(k):
    """Linearised Hall-MHD operator L(k) acting on (rho', v, p', b) (reference)."""
    ik = 1j * np.asarray(k, float)
    K, C = np.cross(ik, np.eye(3)).T, np.cross(B0_vec, np.eye(3)).T.astype(complex)
    L = np.zeros((8, 8), complex)
    L[0, 1:4] = -rho_0 * ik
    L[1:4, 4] = -ik / rho_0
    L[1:4, 5:8] = (-np.outer(ik, B0_vec) + (ik @ B0_vec) * np.eye(3)) / rho_0
    L[4, 1:4] = -gamma * p_0 * ik
    L[5:8, 1:4] = -K @ C
    L[5:8, 5:8] = (eta_H / rho_0) * K @ C @ K
    return L


def eigenmodes(k):
    """Positive frequencies (ascending) and eigenvectors of L(k) (reference and initial states)."""
    lam, vectors = np.linalg.eig(linear_operator(k))
    omega = (1j * lam).real
    keep = [i for i in np.argsort(omega) if omega[i] > 1e-9]
    return omega[keep], vectors[:, keep]


def dispersion_roots(k_abs, theta):
    """Positive roots of the closed-form oblique Hall-MHD dispersion relation (Hameiri et al. 2005)."""
    k2, kp2 = k_abs**2, (k_abs * np.cos(theta)) ** 2
    lhs = np.polymul([1.0, -kp2 * v_A**2], [1.0, -k2 * (v_A**2 + c_s**2), k2 * kp2 * v_A**2 * c_s**2])
    rhs = kp2 * k2 * v_A**2 * d_i**2 * np.array([1.0, -k2 * c_s**2, 0.0])
    w2 = np.roots(np.polysub(lhs, rhs))
    w2 = w2.real[(np.abs(w2.imag) < 1e-9 * np.abs(w2).max()) & (w2.real > 1e-14)]
    return np.sort(np.sqrt(w2))


wavevectors = np.array([k for k, _ in modes], float)
branches = [branch for _, branch in modes]
k_abs = np.linalg.norm(wavevectors, axis=1)
theta = np.arccos(np.abs(wavevectors[:, 2]) / k_abs)
omega_exact = np.array([eigenmodes(k)[0][b] for k, b in zip(wavevectors, branches)])

print(" mode (kx,ky,kz)   |k|    theta    branch               omega (eigenvalue)   omega (closed form)")
for m, (k, branch) in enumerate(modes):
    print(f" {str(k):15s} {k_abs[m]:5.3f}  {np.degrees(theta[m]):5.1f} deg  {branch_name[branch]:18s}"
          f"  {omega_exact[m]:12.7f}        {dispersion_roots(k_abs[m], theta[m])[branch]:12.7f}")

**Runs.** Each run starts from the uniform state plus the real parts of the fourteen eigenvectors
(normalised to $|\hat{\mathbf b}|=10^{-7}$), built with the solver's `conservative_state`. While
the solver advances the state it records, every `sample_interval`, the complex Fourier amplitude
of $\mathbf B$ at each launched wave vector (`fourier_coefficient`), the total energy and the
largest $|\nabla\cdot\mathbf B|$. The amplitude of each mode is the projection of that Fourier
amplitude onto the launched polarisation vector; `oscillation_fit` returns its frequency (slope
of the unwrapped phase, least squares) and its amplitude change.

Main run: spectral, $32^3$, $\Delta t=0.008$. Resolution study: spectral on $8^3$ and $16^3$ with
the same step, central differences on $16^3$, $32^3$, $64^3$ with $\Delta t=0.02$. Time-step
study: spectral $16^3$ with $\Delta t=0.04$, 0.02, 0.01. The difference runs are launched with
the eigenvectors of $L(\tilde{\mathbf k})$, $\tilde k_j=\sin(k_jh)/h$, so each excites a single
discrete branch whose exact frequency is $\omega(\tilde{\mathbf k})$.

## Solve

In [ ]:
configurations = [("spectral", 32, 0.008), ("spectral", 8, 0.008), ("spectral", 16, 0.008),
                  ("spectral", 16, 0.04), ("spectral", 16, 0.02), ("spectral", 16, 0.01),
                  ("centered_fdm", 16, 0.02), ("centered_fdm", 32, 0.02), ("centered_fdm", 64, 0.02)]
runs = {}
for scheme, N, dt in configurations:
    solver = ResistiveMHD3DSolver(mesh_size=N, gamma=gamma, resistivity=0.0, hall_coefficient=eta_H,
                                  final_time=final_time, time_step=dt, time_method="classical_rk4",
                                  derivative_scheme=scheme)
    h = 2 * np.pi / N
    X = np.meshgrid(*solver.axes, indexing="ij")
    k_grid = np.sin(wavevectors * h) / h if scheme == "centered_fdm" else wavevectors
    launched = [eigenmodes(k) for k in k_grid]
    vectors = np.array([v[:, b] / np.linalg.norm(v[5:8, b]) for (_, v), b in zip(launched, branches)])
    waves = np.exp(1j * np.einsum("mi,i...->m...", wavevectors, np.array(X)))
    perturbation = amplitude * np.einsum("mq,m...->q...", vectors, waves).real
    state = solver.conservative_state(rho_0 + perturbation[0], perturbation[1:4], p_0 + perturbation[4],
                                      [B0_vec[i] + perturbation[5 + i] for i in range(3)])
    start = time.perf_counter()
    result = solver.run(state, observe_interval=sample_interval, observe=lambda snapshot: (
        np.array([fourier_coefficient(snapshot.magnetic_field, k, solver.axes) for k in wavevectors]),
        snapshot.total_energy.sum(), np.abs(snapshot.divergence).max()))
    times = np.array([t for t, _ in result.history])
    coefficients = np.array([record[0] for _, record in result.history])
    energy = np.array([record[1] for _, record in result.history])
    polarisation = vectors[:, 5:8]
    signals = np.einsum("tmc,mc->tm", coefficients, polarisation.conj()) / np.sum(np.abs(polarisation) ** 2, 1)
    fits = [oscillation_fit(times, signals[:, m]) for m in range(len(modes))]
    runs[scheme, N, dt] = dict(
        N=N, h=h, dt=dt, times=times, signals=signals, coefficients=coefficients, polarisation=polarisation,
        omega=np.array([f.frequency for f in fits]), amplitude_ratio=np.array([f.amplitude_ratio for f in fits]),
        omega_discrete=np.array([w[b] for (w, _), b in zip(launched, branches)]),
        energy_drift=np.max(np.abs(energy / energy[0] - 1)),
        div_b=max(record[2] for _, record in result.history), final=result.magnetic_field, axes=solver.axes)
    print(f"{'spectral' if scheme == 'spectral' else 'central differences':19s} N = {N:2d}, dt = {dt:.3f}:"
          f" {result.step_count:5d} RK4 steps in {time.perf_counter() - start:6.1f} s")

main = runs["spectral", 32, 0.008]
spectral_study = [runs["spectral", N, 0.008] for N in (8, 16, 32)]
time_study = [runs["spectral", 16, dt] for dt in (0.04, 0.02, 0.01)]
fd_study = [runs["centered_fdm", N, 0.02] for N in (16, 32, 64)]

## Comparison with the reference

**Frequencies** of the main run (spectral, $32^3$, $\Delta t=0.008$) against the exact
dispersion relation, with the polarisation $b_1/b_2$ in the frame $e_1\perp(\mathbf k,\mathbf B_0)$,
$e_2\perp\mathbf k$ in the $(\mathbf k,\mathbf B_0)$ plane: a purely imaginary ratio
$\pm i\,|b_1/b_2|$ means elliptical polarisation, its sign gives the sense of rotation, and the
value is printed as measured (time average) and from the eigenvector.

In [ ]:
khat = wavevectors / k_abs[:, None]
e1 = np.cross(khat, B0_vec)
e1[np.linalg.norm(e1, axis=1) < 1e-12] = (1.0, 0.0, 0.0)
e1 /= np.linalg.norm(e1, axis=1)[:, None]
e2 = np.cross(khat, e1)
measured_ratio = np.einsum("tmc,mc->tm", main["coefficients"], e1) / np.einsum("tmc,mc->tm", main["coefficients"], e2)
theory_ratio = np.einsum("mc,mc->m", main["polarisation"], e1) / np.einsum("mc,mc->m", main["polarisation"], e2)

print(" mode         theta   branch               omega measured   omega exact     rel. dev.  "
      " b1/b2 measured        b1/b2 theory       |a(T)/a(0)|-1")
for m, (k, branch) in enumerate(modes):
    pol = measured_ratio[:, m].mean()
    print(f" {str(k):11s} {np.degrees(theta[m]):5.1f}  {branch_name[branch]:18s}"
          f"  {main['omega'][m]:13.8f}  {omega_exact[m]:13.8f}   {main['omega'][m]/omega_exact[m]-1:+.2e}"
          f"   {pol.real:+.4f}{pol.imag:+.4f}i   {theory_ratio[m].real:+.4f}{theory_ratio[m].imag:+.4f}i"
          f"   {main['amplitude_ratio'][m]-1:+.1e}")
spread = np.max(np.abs(measured_ratio - measured_ratio.mean(0)) / (1 + np.abs(measured_ratio.mean(0))))
print(f"\nlargest change of b1/b2 during the run, relative to 1 + |b1/b2|: {spread:.1e}")
print(f"total energy: largest relative change {main['energy_drift']:.1e};  max |div B| = {main['div_b']:.1e}")

**Resolution study.** The spectral derivative is exact for every resolved Fourier mode, so the
spectral frequencies should not change with $N$ (all three spectral runs use $\Delta t=0.008$;
what remains is the RK4 time error). The central-difference frequencies converge to
the exact value with second order and must equal the modified-wavenumber prediction
$\omega(\tilde{\mathbf k})$ on each grid. From the three grids $N=16,32,64$ the observed order is
$p=\log_2[(\omega_{16}-\omega_{32})/(\omega_{32}-\omega_{64})]$ and the Richardson estimate
$\omega_{64}+(\omega_{64}-\omega_{32})/(2^p-1)$, which uses no knowledge of the exact value.
The difference runs use $\Delta t=0.02$, so their agreement with $\omega(\tilde{\mathbf k})$ is
limited by the RK4 time error of a few $10^{-6}$ (see the time-step study).

In [ ]:
print("\nlargest relative deviation from the exact frequency over all 14 modes")
for run in spectral_study:
    print(f"  spectral            N = {run['N']:2d}: {np.max(np.abs(run['omega'] / omega_exact - 1)):.2e}")
for run in fd_study:
    print(f"  central differences N = {run['N']:2d}: {np.max(np.abs(run['omega'] / omega_exact - 1)):.2e}"
          f"   (vs modified-wavenumber prediction: {np.max(np.abs(run['omega'] / run['omega_discrete'] - 1)):.1e})")

print("\ncentral differences, per mode: omega on N = 16, 32, 64, observed order, Richardson estimate vs exact")
w = np.array([run["omega"] for run in fd_study])
order = np.log2((w[0] - w[1]) / (w[1] - w[2]))
richardson = w[2] + (w[2] - w[1]) / (2**order - 1)
for m, (k, branch) in enumerate(modes):
    print(f"  {str(k):11s} {branch_name[branch]:18s} " + "  ".join(f"{v:.6f}" for v in w[:, m])
          + f"   p = {order[m]:.2f}   Richardson {richardson[m]:.6f} ({richardson[m]/omega_exact[m]-1:+.1e})"
          f"   exact {omega_exact[m]:.6f}")

**Time-step study** (spectral, $16^3$): the error of RK4 must fall with the fourth power of
$\Delta t$ until it reaches the level set by the small nonlinearity of the waves.

In [ ]:
print("\nspectral 16^3: largest relative frequency deviation over all modes")
errors = np.array([np.max(np.abs(run["omega"] / omega_exact - 1)) for run in time_study])
for i, run in enumerate(time_study):
    observed = f"   observed order {np.log2(errors[i-1]/errors[i]):.2f}" if i else ""
    print(f"  dt = {run['dt']:.3f}: {errors[i]:.2e}{observed}")
fastest = int(np.argmax(omega_exact))
print(f"  (dominated by the fastest mode {modes[fastest][0]}, omega dt = "
      f"{omega_exact[fastest] * time_study[0]['dt']:.2f} at the largest step)")

## Figure

Top: two slices of the computed magnetic perturbation at $t=4\pi$ (spectral, $32^3$, shown
with spectral interpolation): $b_x$ in the plane $z=0$ perpendicular to $\mathbf B_0$ and $b_y$
in the plane $y=0$ containing $\mathbf B_0$ (along the vertical axis), and the time series of
two launched modes against $\cos\omega t$ from the dispersion relation. Bottom: the dispersion
relation, frequency against the angle $\theta$ for $|\mathbf k|=3$ and against $|\mathbf k|$ for
$\theta=0^\circ$ and $45^\circ$ (lines: closed form, symbols: measured), and the resolution and
time-step studies.

In [ ]:
fig = plots.figure(layout=[["bx", "by", "series", "series"], ["angle", "wavenumber", "resolution", "time step"]],
                   size=(16, 9.2))

fine = np.linspace(0, 2 * np.pi, 160, endpoint=False)
fig["bx"].image(fine, fine, fourier_interpolate(main["final"][0][:, :, 0], 160) / amplitude,
                colormap="diverging", label="b / (10^-7 B_0)")
fig["bx"].labels("x", "y", title="b_x on the plane z = 0, normal to B_0")
fig["by"].image(fine, fine, fourier_interpolate(main["final"][1][:, 0, :], 160) / amplitude,
                colormap="diverging", label="b / (10^-7 B_0)")
fig["by"].labels("x", "z", title="b_y on the plane y = 0, containing B_0")

panel = fig["series"]
t_fine = np.linspace(0, final_time, 1200)
for colour, (m, every) in enumerate(((6, 1), (13, 2))):
    a = main["signals"][:, m] / main["signals"][0, m]
    panel.reference(t_fine, np.cos(omega_exact[m] * t_fine), color=colour)
    panel.points(main["times"][::every], a.real[::every], color=colour,
                 label=f"k = {modes[m][0]}, {branch_name[branches[m]]}, omega = {main['omega'][m]:.4f}")
panel.labels("time t", "Re a(t) / a(0)",
             title="Launched-mode amplitudes (symbols) and cos(omega t) from the dispersion relation")
panel.legend("lower left")

panel = fig["angle"]
angles = np.linspace(0.0, np.pi / 2, 181)
roots = np.array([np.r_[np.nan, np.nan, np.nan, dispersion_roots(3.0, th)][-3:] for th in angles])
on_shell = np.isclose(k_abs, 3.0)
panel.line(np.degrees(angles), roots[:, 2], "fast / whistler", color=1)
panel.line(np.degrees(angles), roots[:, 1], "ion-cyclotron / KAW", color=0)
panel.line(np.degrees(angles), roots[:, 0], "slow", color=2)
panel.points(np.degrees(theta[on_shell]), main["omega"][on_shell], "measured", color="ink")
panel.labels("angle theta between k and B_0 [deg]", "omega", title="|k| = 3: frequency vs angle")

panel = fig["wavenumber"]
k_line = np.linspace(0.05, 4.6, 200)
for degrees, style in ((0.0, "solid"), (45.0, "dashed")):
    r = np.array([dispersion_roots(k, np.radians(degrees)) for k in k_line])
    lower = k_line * v_A * (np.sqrt(1 + (k_line * d_i / 2) ** 2) - k_line * d_i / 2) if degrees == 0 else r[:, -2]
    panel.line(k_line, r[:, -1], f"whistler, theta = {degrees:.0f} deg", color=1, style=style)
    panel.line(k_line, lower, f"ion-cyclotron/KAW, theta = {degrees:.0f} deg", color=0, style=style)
on_lines = np.isclose(np.degrees(theta), 0.0) | np.isclose(np.degrees(theta), 45.0)
panel.line(k_line, k_line * v_A, "Alfven omega = k v_A", color="ink", style="dotted")
panel.points(k_abs[on_lines], main["omega"][on_lines], "measured", color="ink")
panel.labels("|k|", "omega", title="Frequency vs |k| (symbols: measured)").legend("upper left")

panel = fig["resolution"]
hs = [run["h"] for run in fd_study]
for m in (4, 6, 11):
    panel.line(hs, np.abs(w[:, m] / omega_exact[m] - 1), f"central diff., k = {modes[m][0]}", markers=True)
panel.slope(2)
panel.line([run["h"] for run in spectral_study], [np.max(np.abs(run["omega"] / omega_exact - 1)) for run in spectral_study],
           "spectral, all modes (max), dt = 0.008", markers=True)
panel.scale("log", "log").labels("grid spacing h", "|omega / omega_exact - 1|", title="Resolution study")

panel = fig["time step"]
panel.line([run["dt"] for run in time_study], errors, "spectral 16^3, max over modes", markers=True).slope(4)
panel.scale("log", "log").labels("time step dt", "|omega / omega_exact - 1|", title="Time-step study (RK4)")
plots.show()

## Going further

Raise the amplitude to $10^{-2}B_0$ to see the nonlinear frequency shift and the transfer of
energy between modes that share wave-vector triads, or set `eta_H = 0` to recover the ideal
Alfvén and magnetosonic waves (the whistler and ion-cyclotron branches then merge into the
shear-Alfvén wave $\omega=k_\parallel v_A$). A finite `resistivity` adds the damping rate
$\eta k^2/2$ to every mode, which the same phase-and-amplitude fit measures directly.